In [1]:
import os
os.chdir("..") # set parent dir as root dir

<img src="../src/img/rag_architecture.png"/>

In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

langsmith_api_key = os.getenv("LANGSMITH_API_KEY", None)
if not langsmith_api_key:
    raise ValueError("LangSmith API key is missing in .env")
os.environ["LANGCHAIN_API_KEY"] = langsmith_api_key

openrouter_api_key = os.getenv("OPENROUTER_API_KEY", None)
if not openrouter_api_key: 
    raise ValueError("Missing OpenRouter API key value in env.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

##### Tokenization overview

In [3]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

tiktoken

In [4]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

print(num_tokens_from_string(string=question, encoding_name="cl100k_base"))
print(num_tokens_from_string(string=document, encoding_name="cl100k_base"))

8
7


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "BAAI/bge-small-en-v1.5"
embd = HuggingFaceEmbeddings(model_name=embedding_model_name)

query_result = embd.embed_query(question)
document_result = embd.embed_query(document)

print(len(query_result))
print(len(document_result))

384
384


In [6]:
print(type(query_result))

<class 'list'>


In [7]:
import numpy as np

def cosine_simularity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    vec1_norm = np.linalg.norm(vec1)
    vec2_norm = np.linalg.norm(vec2)
    return dot_product / (vec1_norm * vec2_norm)

In [8]:
simularity = cosine_simularity(query_result, document_result)
print(f"Cosin simularity: {simularity}")

Cosin simularity: 0.737882228232248


### Part 1: Indexing & Retriever

<img src="../src/img/indexing.png"/>

##### Load data

In [9]:
import bs4 # for parsing
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict( # kwargs for BeautifulSoup
        parse_only=bs4.SoupStrainer( # add bs4 filter
            class_=("post-content", "post-title", "post-header")
            # parse only only HTML elems with class attribute post-content, 
            # post-title or post-header
        )
    ),
)
blog_docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


##### Split data on small documents

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50, 
)

# Make splits
splits = text_splitter.split_documents(blog_docs)

##### Vectorstore

In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embd, 
)

##### Retreiver

Retreiver creation

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={
    "k": 1, # find 1 nearest neighbor (1 doc)
})

Retreiver use

In [13]:
docs = retriever.invoke("What is Task Decomposition?") # list of docs

In [14]:
print(docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a

In [15]:
print(docs[0].page_content)

Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) by LLM with simple prompt

### Step 2: Generation

<img src="../src/img/generation.png"/>

In [16]:
from langchain_openai import ChatOpenAI

openrouter_model = os.getenv("OPENROUTER_MODEL", None)
if not openrouter_model:
    raise ValueError("Missing openrouter model name in env.")

llm = ChatOpenAI(
    model=openrouter_model, 
    base_url="https://openrouter.ai/api/v1", 
    api_key=openrouter_api_key, 
    temperature=0, 
)

In [26]:
response = llm.invoke(input="What is Task Decomposition?")

In [27]:
print(response)

content='Task decomposition is the process of breaking down a complex task or problem into smaller, more manageable sub-tasks. It\'s a fundamental technique used in various fields, including project management, software development, problem-solving, and even everyday life.\n\nHere\'s a breakdown of what it entails:\n\n**Key Concepts:**\n\n* **Complex Task:** The initial, large, and often overwhelming task that needs to be accomplished.\n* **Sub-tasks:** The smaller, more focused, and easier-to-understand components that the complex task is divided into.\n* **Hierarchy:**  Sub-tasks can be further broken down into even smaller sub-tasks, creating a hierarchical structure. This allows for a more granular approach to managing complexity.\n* **Dependencies:**  Often, sub-tasks have dependencies on each other.  Some tasks need to be completed before others can begin. Understanding these dependencies is crucial for effective task decomposition.\n\n**Why is Task Decomposition Important?**\n\n

In [28]:
print(response.content)

Task decomposition is the process of breaking down a complex task or problem into smaller, more manageable sub-tasks. It's a fundamental technique used in various fields, including project management, software development, problem-solving, and even everyday life.

Here's a breakdown of what it entails:

**Key Concepts:**

* **Complex Task:** The initial, large, and often overwhelming task that needs to be accomplished.
* **Sub-tasks:** The smaller, more focused, and easier-to-understand components that the complex task is divided into.
* **Hierarchy:**  Sub-tasks can be further broken down into even smaller sub-tasks, creating a hierarchical structure. This allows for a more granular approach to managing complexity.
* **Dependencies:**  Often, sub-tasks have dependencies on each other.  Some tasks need to be completed before others can begin. Understanding these dependencies is crucial for effective task decomposition.

**Why is Task Decomposition Important?**

* **Improved Manageabili